# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook shows how to explore the FAIR^2 colorectal cancer survivor dataset using the `mlcroissant` library and the Croissant standard metadata.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as one object)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, their IDs, and the fields (columns) available in each. All entities are referenced by their `@id`.

In [ ]:
# List available record sets and their IDs
record_sets = list(dataset.record_sets.values())
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    # List fields for this record set
    print(f"  Fields (@id):")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) type: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from record set(s) into pandas DataFrames for analysis. Use the proper record set and field `@id`s as displayed above.

In [ ]:
# Extract data from all record sets into DataFrames, using their `@id`
df_dict = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        df_dict[rs.id] = df
        print(f"Loaded record set '{rs.name}' with {len(df)} records. Columns:")
        print(list(df.columns))
        print()

# Pick first record set for demonstration
record_set_id = record_sets[0].id
df = df_dict[record_set_id]
df.head()

## 4. Exploratory Data Analysis (EDA)
Examples of possible EDA: filtering records, normalization of numeric fields, grouping/categorizing.

All field columns below are selected and referenced by their `@id` as per best practice.

In [ ]:
# Example: Analyze the field representing patient age (@id: 'age' or similar)
# First, display all field @ids and pick a numeric one for demo
print("All DataFrame columns:")
print(df.columns.tolist())

# Let's try to use an age column, if available, otherwise pick the first numeric field
numeric_field_id = None
possible_age_aliases = ['age', 'schema:age', 'cr:age', 'http://schema.org/age']
for col in df.columns:
    if str(col).lower() in possible_age_aliases:
        numeric_field_id = col
        break

# Fallback heuristic: find first column with likely numeric values
if numeric_field_id is None:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Selected numeric field for normalization and filtering: {numeric_field_id}")

# Filter records where value in this field is above a threshold (example: age > 50)
threshold = 50

filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize this field (Z-score normalization)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by a categorical field (e.g., 'sex', 'anatomical location', find suitable field by id)
group_field = None
possible_group_fields = ['sex', 'gender', 'anatomical_location', 'cr:anatomical_location', 'schema:sex', 'schema:gender']
for col in df.columns:
    if str(col).lower() in possible_group_fields:
        group_field = col
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field}:")
    print(grouped_df.head())
else:
    print("\nNo obvious group field found.")

## 5. Visualization
Visualize the data distribution for the selected numeric field and optionally faceted/grouped by the group field, using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
if group_field:
    sns.histplot(data=filtered_df, x=numeric_field_id, hue=group_field, kde=True, multiple='stack')
    plt.title(f"Distribution of {numeric_field_id} by {group_field} (filtered)")
else:
    sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (filtered)")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, we've demonstrated access and exploration of the FAIR^2 colorectal cancer survivor dataset using `mlcroissant`. We've shown how to load record sets by `@id`, list their fields, extract records, preprocess numeric fields, and visualize their distributions. This workflow provides a reproducible way to work with Croissant datasets for downstream analysis and ML tasks.